# CalMS21

Understand video lengths, behavior distributions, tracking confidence, and label sequences.

The training set contains **70 videos and 507,738 frames**. The labels are imbalanced: **other 62.72%**, **investigation 28.88%**, **mount 5.64%**, and **attack 2.77%**. 

## Dataset background

- `keypoints`: `(frames, 2 mice, 2 coordinates, 7 body parts)`, in pixels within a 1024 × 570 image.
- `scores`: `(frames, 2 mice, 7 body parts)`, tracking confidence between 0 and 1.
- `annotations`: one behavior label per frame.
- Mouse 0 is the resident; mouse 1 is the intruder.
- Body parts: nose, left ear, right ear, neck, left hip, right hip, tail base.

Install the packages in terminal or in notebook:
```sh
pip install ijson ipykernel # if on venv
!pip install ijson ipykernel # in notebook
```

The JSON is large, so `ijson` reads one video at a time.

### 1. Import packages

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ijson
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path.cwd()
ROOT = ROOT.parent
RAW = ROOT / 'data/raw/task1_classic_classification'
POSE_PATH = RAW / 'calms21_task1_train.json'
FEATURE_PATH = RAW / 'taskprog_features_task1_train.json'

VOCAB = {'attack': 0, 'investigation': 1, 'mount': 2, 'other': 3}
CONFIDENCE_THRESHOLD = 0.5  # temporary threshold
SEED = 42               
COLORS = ['#3569a8', '#c88c24', '#b75a42', '#6b727c']
plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

### 2. List files and read one video at a time

In [ ]:
# Show file sizes
files = []
for path in sorted(RAW.glob('*.json')):
    files.append({'file': path.name, 'size_MiB': path.stat().st_size / 2**20})
display(pd.DataFrame(files).round(1))


#### Reusable helpers

These functions handle repeated loading, array conversion, keypoint checks, example selection, and pose drawing.

In [ ]:
def iter_videos(path):
    """Yield one video ID and data dictionary at a time from a Task 1 JSON file"""
    with path.open('rb') as stream:
        yield from ijson.kvitems(stream, 'annotator-id_0', use_float=True)


def get_video_arrays(video):
    """Return a pose video's keypoints, confidence scores, and labels as np arrays"""
    keypoints = np.asarray(video['keypoints'], dtype=np.float32)
    scores = np.asarray(video['scores'], dtype=np.float32)
    labels = np.asarray(video['annotations'], dtype=np.int64)
    return keypoints, scores, labels


def valid_keypoints(keypoints, scores, threshold=None):
    """Return a per-keypoint mask for finite, in-image coordinates and optional confidence filtering"""
    x = keypoints[:, :, 0, :]
    y = keypoints[:, :, 1, :]
    valid = (np.isfinite(x) & np.isfinite(y) & np.isfinite(scores)
             & (x >= 0) & (x <= 1024) & (y >= 0) & (y <= 570))
    
    if threshold is not None:
        valid &= scores >= threshold
    return valid


def get_video_example(path, video_id=None, max_frames=None):
    """Return the first or a specified video, optionally keeping only its first max_frames frames."""
    videos = iter_videos(path)
    try:
        for current_id, video in videos:
            if video_id is not None and current_id != video_id:
                continue
            if max_frames is not None:
                # Metadata stays intact
                video = {field: values[:max_frames] if field in
                         ('keypoints', 'scores', 'annotations', 'features') else values
                         for field, values in video.items()}
            return current_id, video
    finally:
        videos.close()  # Close the file even when returning before the scan ends
    raise ValueError(f'Video {video_id!r} was not found in {path.name}')


def plot_pose(ax, pose, title, crop_size):
    """Draw both mice's numbered keypoints and body connections on an equally scaled image crop"""
    connections = [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4), (3, 5), (4, 6), (5, 6)]
    mice = [('Resident', COLORS[0], 'o'), ('Intruder', COLORS[1], '^')]
    for mouse, (name, color, marker) in enumerate(mice):
        x, y = pose[mouse]
        for start, end in connections:
            ax.plot(x[[start, end]], y[[start, end]], color=color, linewidth=1, alpha=0.7)
        ax.scatter(x, y, color=color, marker=marker, s=45, label=name, zorder=3)
        for part in range(7):
            ax.annotate(str(part + 1), (x[part], y[part]), xytext=(5, 5),
                        textcoords='offset points', fontsize=9, color=color)

    center_x = (pose[:, 0, :].min() + pose[:, 0, :].max()) / 2
    center_y = (pose[:, 1, :].min() + pose[:, 1, :].max()) / 2
    ax.set_xlim(center_x - crop_size / 2, center_x + crop_size / 2)
    ax.set_ylim(center_y + crop_size / 2, center_y - crop_size / 2)
    ax.set_aspect('equal')
    ax.set(title=title, xlabel='Image x (pixels)', ylabel='Image y (pixels)')
    ax.legend(loc='upper right', fontsize=9)


### 3. Inspect one video

In [ ]:
# Keep a small example rather than a whole video's nested lists
video_id, sample = get_video_example(POSE_PATH, max_frames=1000)
print(f"Example video: {video_id} (up to the first 1,000 frames)")
print(f"Fields: {list(sample)}")
print(f"Metadata: {sample['metadata']}")
for field in ['keypoints', 'scores', 'annotations']:
    print(f"{field} {np.asarray(sample[field]).shape}")

### 4. Summarize all videos

A keypoint is flagged if:
- confidence is below 0.5 
- coordinate is nonfinite
- it outside the image

In [ ]:
rows = []
bouts = []
confidence_edges = np.linspace(0, 1, 21)  # Twenty bins over the confidence range
confidence_hist = np.zeros((2, 7, 20), dtype=np.int64)

for video_id, video in iter_videos(POSE_PATH):
    keypoints, scores, labels = get_video_arrays(video)
    frame_count = len(labels)
    flagged = ~valid_keypoints(keypoints, scores, CONFIDENCE_THRESHOLD)

    # Count frames for each of the four behaviors in the video
    counts = np.bincount(labels, minlength=4)
    row = {
        'video_id': video_id, 
        'identity': re.search(r'mouse\d+', video_id).group(),
        'frames': frame_count, 
        'flagged_keypoint_pct': 100 * flagged.mean(),
        'nonfinite_coordinates': int((~np.isfinite(keypoints)).sum())
    }
    row.update(dict(zip(VOCAB, counts)))
    rows.append(row)

    # Accumulate confidence histograms
    for mouse in range(2):
        for part in range(7):
            confidence_hist[mouse, part] += np.histogram(scores[:, mouse, part], confidence_edges)[0]

    # Label changes mark the start of a new behavior 
    changes = np.flatnonzero(labels[1:] != labels[:-1]) + 1
    boundaries = np.r_[0, changes, frame_count]
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        bouts.append((int(labels[start]), int(end - start)))

example_id, example_data = get_video_example(POSE_PATH, max_frames=1000)
example = {'video_id': example_id, 'labels': np.asarray(example_data['annotations'])}

audit = pd.DataFrame(rows)
print(f'{len(audit)} videos; {audit.frames.sum():,} labeled frames')
display(audit.head().round(2))
display(audit[['frames', 'flagged_keypoint_pct', 'nonfinite_coordinates']].describe().round(2))

### 5. Class balance and behavior duration

In [ ]:
# Add up frame counts across all training videos
class_counts = audit[list(VOCAB)].sum()
class_table = pd.DataFrame({'frames': class_counts, 'percent': 100 * class_counts / class_counts.sum(), 'videos_with_class': (audit[list(VOCAB)] > 0).sum()})
display(class_table.round(2))

# Plot percentages so the class imbalance is easy to see
fig, ax = plt.subplots()
ax.bar(class_table.index, class_table.percent, color=COLORS)
ax.set(title='Behavior balance', ylabel='Share of frames (%)')
for i, value in enumerate(class_table.percent):
    ax.text(i, value + 1, f'{value:.1f}%', ha='center')
ax.set_ylim(0, class_table.percent.max() + 10)
fig.tight_layout()
plt.show()

#### Behavior duration

Measure consecutive runs in frames. The log axis makes short and long runs visible together.

In [ ]:
# A bout is a consecutive run of frames with the same label
bout_table = pd.DataFrame(bouts, columns=['label', 'frames'])

fig, ax = plt.subplots()
ax.boxplot([bout_table.loc[bout_table.label == label, 'frames'] for label in VOCAB.values()],
           labels=list(VOCAB), showfliers=False)
ax.set(yscale='log', title='Continuous behavior runs', ylabel='Run length (frames, log scale)') # log scale
fig.tight_layout()
plt.show()

display(bout_table.groupby('label').frames.agg(['count', 'median', 'max']).rename(index={v:k for k,v in VOCAB.items()}))

#### Example poses: 14 keypoints for each behavior

**1 nose, 2 left ear, 3 right ear, 4 neck, 5 left hip, 6 right hip, 7 tail base**

In [ ]:
# Find one clearly tracked example for each behavior, one video at a time
pose_examples = {}
for video_id, video in iter_videos(POSE_PATH):
    poses, confidence, frame_labels = get_video_arrays(video)

    # Rank valid frames by the least-confident point across both mice
    frame_quality = confidence.min(axis=(1, 2))
    valid = valid_keypoints(poses, confidence).all(axis=(1, 2))
    for behavior, label in VOCAB.items():
        candidates = np.flatnonzero((frame_labels == label) & valid)
        if len(candidates) == 0:
            continue
        frame = candidates[np.argmax(frame_quality[candidates])]
        if behavior not in pose_examples or frame_quality[frame] > pose_examples[behavior]['confidence']:
            pose_examples[behavior] = {'pose': poses[frame].copy(), 'video_id': video_id, 'frame': int(frame), 'confidence': float(frame_quality[frame])}

#### Draw the selected poses

Use the same crop size in all four panels so distances stay comparable.

In [ ]:
# Use the same crop width and height in every panel
crop_size = max(np.ptp(item['pose'], axis=(0, 2)).max() for item in pose_examples.values()) + 80
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, behavior in zip(axes.flat, VOCAB):
    item = pose_examples[behavior]
    title = f"{behavior.capitalize()} — frame {item['frame']}\n{item['video_id'].split('/')[-1]}"
    plot_pose(ax, item['pose'], title, crop_size)

fig.suptitle('Example poses by behavior — 7 keypoints per mouse', fontsize=14)
fig.tight_layout()
plt.show()

display(pd.DataFrame([{'behavior': behavior, 'video_id': item['video_id'], 'frame': item['frame'], 'minimum_point_confidence': item['confidence']}
                      for behavior, item in pose_examples.items()]).round(3))

### 6. Explore confidence and an example label timeline

The confidence chart summarizes all training keypoints. The timeline shows only the first 1,000 frames of one video

In [ ]:
# Use the center of each confidence bin for the x-axis
centers = (confidence_edges[:-1] + confidence_edges[1:]) / 2
fig, ax = plt.subplots()

# Compare resident and intruder tracking confidence
for mouse, style in enumerate(['-', '--']):
    # Combine the seven body parts for this mouse
    hist = confidence_hist[mouse].sum(axis=0)
    ax.plot(centers, hist / hist.sum() * 100, style, label=['resident', 'intruder'][mouse], color=COLORS[mouse])

ax.axvline(CONFIDENCE_THRESHOLD, color='#333333', linestyle=':', label='initial mask threshold')
ax.set(title='Tracking confidence', xlabel='Confidence score', ylabel='Keypoints per bin (%)')
ax.legend() 

fig.tight_layout() 
plt.show()

#### Label timeline

Show labels in their original frame order for one video.

In [ ]:
# Show consecutive frame labels from one video without shuffling them
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.scatter(np.arange(len(example['labels'])), example['labels'], c=[COLORS[v] for v in example['labels']], s=4)
ax.set(yticks=list(VOCAB.values()), yticklabels=list(VOCAB), xlabel='Frame index', title=f"Label timeline — {example['video_id'].split('/')[-1]} (first 1,000 frames)")
fig.tight_layout() 
plt.show()

### 7. Video split with close class proportions

In [ ]:
# One row per video makes it easy to split complete recordings
video_ids = audit.video_id.to_numpy()
groups = audit.identity.to_numpy()

class_counts = audit[list(VOCAB)].to_numpy()
overall_share = class_counts.sum(axis=0) / class_counts.sum()
splitter = GroupShuffleSplit(n_splits=500, test_size=0.2, random_state=SEED)
best_score = np.inf

for candidate_train, candidate_validation in splitter.split(video_ids, groups=groups):
    train_counts = class_counts[candidate_train].sum(axis=0)
    validation_counts = class_counts[candidate_validation].sum(axis=0)

    if (train_counts == 0).any() or (validation_counts == 0).any():
        continue  # Both splits must include every class
    
    train_share = train_counts / train_counts.sum()
    validation_share = validation_counts / validation_counts.sum()

    # Relative errors give rare behaviors equal attention to common behaviors
    class_error = (np.abs(train_share - overall_share) / overall_share).mean()
    class_error += (np.abs(validation_share - overall_share) / overall_share).mean()

    # Aim for validation to contain about 20% of all frames
    frame_error = abs(validation_counts.sum() / class_counts.sum() - 0.2) / 0.2
    score = class_error + frame_error
    if score < best_score:
        best_score = score
        train_index, validation_index = candidate_train, candidate_validation

# No identity group should appear on both sides of the split
split_ids = {'train': video_ids[train_index].tolist(), 'validation': video_ids[validation_index].tolist()}

# Compare behavior percentages within each split
split_rows = []
for split, ids in split_ids.items():
    subset = audit.set_index('video_id').loc[ids]
    counts = subset[list(VOCAB)].sum()

    row = {'split': split, 'videos': len(ids), 'frames': int(subset.frames.sum())}
    row.update((counts / counts.sum() * 100).to_dict())
    split_rows.append(row)

split_summary = pd.DataFrame(split_rows).set_index('split')
display(split_summary.round(2))

#### Compare the split proportions

In [ ]:
# Rows become behavior labels; the two columns become train/validation bars.
ax = split_summary[list(VOCAB)].T.plot.bar(color=COLORS[:2], rot=0)
ax.set(title='Class coverage after holding out video groups',
       xlabel='Behavior', ylabel='Share of split frames (%)')
ax.figure.tight_layout()
plt.show()

### 8. TREBA features 

TREBA provides **32 learned values per frame**, derived from keypoint trajectories. Individual dimensions have no known physical meaning; names F01–F32 below correspond to array columns 0–31.

Join feature rows to labels using video IDs and frame positions. Use only the **training side of step 7's split** for these plots, keeping validation and official test data out of this feature exploration. 

This might be useful as we can feed this directly to a SVM/Random Forest etc.

In [ ]:
# Stream the feature JSON, keeping only videos assigned to training
feature_arrays, label_arrays = [], []
training_videos = set(split_ids['train'])

# Load the labels here
treba_labels = {}
for video_id, record in iter_videos(POSE_PATH):
    if video_id in training_videos:
        treba_labels[video_id] = np.asarray(record['annotations'], dtype=np.int64)

matched_videos = set()
treba_example_id = None

for video_id, video in iter_videos(FEATURE_PATH):
    if video_id not in training_videos:
        continue
    features = np.asarray(video['features'], dtype=np.float32)
    labels = treba_labels[video_id]
    matched_videos.add(video_id)
    feature_arrays.append(features)
    label_arrays.append(labels)
    if treba_example_id is None:
        treba_example_id = video_id

# Retrieve a short segment of the same video for the temporal plot
example_id, example_data = get_video_example(FEATURE_PATH, treba_example_id, max_frames=500)
treba_example = {'video_id': example_id,
                 'features': np.asarray(example_data['features'], dtype=np.float32)}

# Each row is one frame
treba_X = np.concatenate(feature_arrays)
treba_y = np.concatenate(label_arrays)
feature_names = [f'F{i + 1:02d}' for i in range(32)]
print(f'TREBA: {len(matched_videos)} training videos, {len(treba_X):,} frames, 32 features')

# Summarize scale and spread without printing the full feature array
feature_summary = pd.DataFrame(
    {
        'mean': treba_X.mean(axis=0),
        'std': treba_X.std(axis=0),
        'min': treba_X.min(axis=0),
        'max': treba_X.max(axis=0)
    }, 
    index=feature_names
)

display(feature_summary.round(3))
del feature_arrays, label_arrays

### 9. Plot feature distributions

In [ ]:
# The shape, spread, and tails of individual features
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, column in zip(axes.flat, [0, 8, 16, 24]):
    ax.hist(treba_X[:, column], bins=60, color=COLORS[0], density=True)
    ax.set(title=feature_names[column], xlabel='Raw TREBA value', ylabel='Density')
    
fig.suptitle('TREBA distributions — training frames', fontsize=13)
fig.tight_layout()
plt.show()

### 10. Check correlations between dimensions

In [ ]:
correlation = pd.DataFrame(treba_X, columns=feature_names).corr()

fig, ax = plt.subplots(figsize=(9, 8))
color_map = plt.get_cmap('RdBu_r').copy()
image = ax.imshow(correlation, vmin=-1, vmax=1, cmap=color_map)

ticks = np.arange(0, 32, 4)
ax.set(xticks=ticks, xticklabels=np.array(feature_names)[ticks],
       yticks=ticks, yticklabels=np.array(feature_names)[ticks],
       title='TREBA dimension correlations — training frames',
       xlabel='Feature dimension', ylabel='Feature dimension')
fig.colorbar(image, ax=ax, label='Pearson correlation')
fig.tight_layout()
plt.show()

### 11. Compare standardized features across behaviors

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale training features for comparison, leaving raw TREBA values unchanged.
treba_scaler = StandardScaler()
treba_scaled = treba_scaler.fit_transform(treba_X)

# Average each feature within each behavior, keeping the original label order.
feature_table = pd.DataFrame(treba_scaled, columns=feature_names)
class_profiles = feature_table.groupby(treba_y).mean().reindex(list(VOCAB.values()))

#### Plot the average feature profiles

Colors use one symmetric scale measured in training standard deviations.

In [ ]:
# Keep the same color limits for every behavior.
limit = max(float(np.abs(class_profiles.to_numpy()).max()), 0.01)
fig, ax = plt.subplots(figsize=(12, 4))
image = ax.imshow(class_profiles, aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit)
ax.set(xticks=range(32), xticklabels=feature_names,
       yticks=range(4), yticklabels=list(VOCAB),
       title='Mean standardized TREBA profile by behavior',
       xlabel='Feature dimension', ylabel='Behavior')
ax.tick_params(axis='x', labelrotation=90)
fig.colorbar(image, ax=ax, label='Mean value (training standard deviations)')
fig.tight_layout()
plt.show()

# Report how many frames contribute to each behavior's mean.
class_frame_counts = pd.Series(treba_y).value_counts().reindex(list(VOCAB.values()))
class_frame_counts.index = list(VOCAB)
display(class_frame_counts.rename('Training frames'))

### 12. Visualize the feature space with PCA

In [ ]:
from sklearn.decomposition import PCA

# Fit PCA on all standardized training frames without using behavior labels.
treba_pca = PCA(n_components=32, svd_solver='full')
projection = treba_pca.fit_transform(treba_scaled)
variance = treba_pca.explained_variance_ratio_

# Pick display points separately so sampling does not clutter the plotting code.
rng = np.random.default_rng(SEED)
pca_samples = {}
for behavior, label in VOCAB.items():
    indices = np.flatnonzero(treba_y == label)
    pca_samples[behavior] = rng.choice(indices, size=min(1000, len(indices)), replace=False)

#### PCA scatter

In [ ]:
fig, ax = plt.subplots()
for (behavior, sample), color in zip(pca_samples.items(), COLORS):
    ax.scatter(projection[sample, 0], projection[sample, 1],
               s=9, alpha=0.3, color=color, label=behavior)
ax.set(title='PCA view — up to 1,000 frames per behavior',
       xlabel=f'PC1 ({variance[0]:.1%} of variance)',
       ylabel=f'PC2 ({variance[1]:.1%} of variance)')
ax.legend(markerscale=2)
fig.tight_layout()
plt.show()

#### Variance

In [ ]:
fig, ax = plt.subplots()
ax.plot(range(1, 33), 100 * np.cumsum(variance), color=COLORS[0])
ax.axvline(2, linestyle=':', color='#333333', label='2D view')
ax.set(title='Cumulative PCA variance', xlabel='Number of components',
       ylabel='Variance explained (%)', ylim=(0, 101))
ax.legend()
fig.tight_layout()
plt.show()
print(f'The first two components explain {variance[:2].sum():.1%} of standardized training variance.')

### 13. TREBA features over consecutive frames

In [ ]:
# Use the original frame order as this is temporal info
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, column in zip(axes, [0, 8, 16]):
    ax.plot(treba_example['features'][:, column], color=COLORS[0], linewidth=1)
    ax.set_ylabel(f'{feature_names[column]} value')
    
axes[-1].set_xlabel('Frame index')
fig.suptitle(f"TREBA timeline — {treba_example['video_id'].split('/')[-1]}")  # First 500 frames
fig.tight_layout()
plt.show()